# 🎧 3D Spatial Audio with HRTF — CIPIC Database
### Master's Course: Virtual Reality and Immersive Audio

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain how **Head-Related Transfer Functions (HRTFs)** model spatial hearing
2. Load and explore the **CIPIC HRTF database** using Python
3. **Visualize** HRIRs and their spectral properties (HRTF, ITD, ILD)
4. **Synthesize** binaural audio for any position in 3D space
5. Interactively **explore** how position affects perceived sound

---

## 1. Background: How We Localize Sound

Human spatial hearing relies on three main acoustic cues:

| Cue | Description | Dominant range |
|-----|-------------|----------------|
| **ITD** – Interaural Time Difference | Time delay between the sound reaching left vs. right ear | Low frequencies (< 1.5 kHz) |
| **ILD** – Interaural Level Difference | Amplitude difference between ears (head shadow) | High frequencies (> 1.5 kHz) |
| **Spectral cues** | Frequency shaping by pinna, head, torso reflections | All frequencies (elevation & front/back) |

The **HRTF** captures all three cues simultaneously. It describes how a sound from a specific direction in space is filtered by your head and ears before reaching the eardrums. Measuring it for both ears gives a **pair of Head-Related Impulse Responses (HRIRs)** — one for the left, one for the right ear.

**Binaural synthesis**: by convolving any mono signal with a left/right HRIR pair, we trick the auditory system into perceiving the sound as coming from that direction — even through headphones.

```
         Left HRIR
    ┌──────────────────┐
    │  mono signal  ──►│──► Left ear channel
    └──────────────────┘
    ┌──────────────────┐
    │  mono signal  ──►│──► Right ear channel
    └──────────────────┘
         Right HRIR
```

### The CIPIC Database

The [CIPIC database](https://www.ece.ucdavis.edu/cipic/spatial-sound/hrtf-data/) (UC Davis) provides HRIRs measured on **45 human subjects** and two KEMAR dummy heads at **1250 spatial positions**:
- **25 azimuths**: from −80° to +80° (interaural-polar coordinates, positive = right)
- **50 elevations**: from −45° to +230.625° in 5.625° steps (full vertical orbit)
- **HRIR length**: 200 samples at 44100 Hz ≈ 4.5 ms

> **Interaural-polar coordinates**: azimuth is the angle in the interaural plane, elevation is the rotation around the interaural axis. 0°/0° = directly in front, 90° elevation = directly above.

---
## 2. Setup: Install Dependencies and Download Dataset

In [ ]:
# ── Install required packages (run once) ──────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['librosa', 'scipy', 'numpy', 'matplotlib', 'ipywidgets', 'soundfile', 'ipython']:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

print('✅ All packages available')

In [ ]:
# ── Core Imports ─────────────────────────────────────────────────────────────
import os
import io
import warnings
import itertools
import numpy as np
import scipy.io
import scipy.signal
import scipy.spatial
import librosa
import librosa.display
import soundfile as sf
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import ipywidgets as widgets
from IPython.display import display, Audio, HTML

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'figure.facecolor': '#FAFAFA', 'axes.grid': True,
                     'grid.alpha': 0.4})

print('✅ Imports complete')

In [ ]:
# ── Download CIPIC Dataset (subject_021 = KEMAR, small pinnae) ───────────────
# The full CIPIC database is hosted on GitHub (mirror by amini-allight).
# We download only one subject MAT file (~500 KB) for this demo.
# To use a different subject, change SUBJECT below.

import urllib.request
from pathlib import Path

SUBJECT = '021'          # 021 = KEMAR small pinnae (standard reference)
                          # 165 = KEMAR large pinnae
                          # 003,008,009,010,... = human subjects

BASE_URL  = 'https://raw.githubusercontent.com/amini-allight/cipic-hrtf-database/master'
MAT_URL   = f'{BASE_URL}/standard_hrir_database/subject_{SUBJECT}/hrir_final.mat'
LOCAL_DIR = Path(f'cipic_data/subject_{SUBJECT}')
LOCAL_MAT = LOCAL_DIR / 'hrir_final.mat'

LOCAL_DIR.mkdir(parents=True, exist_ok=True)

if not LOCAL_MAT.exists():
    print(f'Downloading subject_{SUBJECT} HRIR data...')
    urllib.request.urlretrieve(MAT_URL, LOCAL_MAT)
    print(f'  → saved to {LOCAL_MAT}')
else:
    print(f'✅ Subject_{SUBJECT} data already present at {LOCAL_MAT}')

# ── Define measurement grid ──────────────────────────────────────────────────
# 25 azimuth angles (degrees, interaural-polar)
AZIMUTHS = np.array([
    -80, -65, -55, -45, -40, -35, -30, -25, -20, -15,
    -10,  -5,   0,   5,  10,  15,  20,  25,  30,  35,
     40,  45,  55,  65,  80
])

# 50 elevation angles (degrees): -45° to +230.625° in steps of 5.625°
ELEVATIONS = -45 + 5.625 * np.arange(50)

FS     = 44100   # Hz  — CIPIC measurement sample rate
N_AZ   = len(AZIMUTHS)    # 25
N_EL   = len(ELEVATIONS)  # 50
N_HRIR = 200              # samples per HRIR tap

print(f'Grid: {N_AZ} azimuths × {N_EL} elevations = {N_AZ*N_EL} positions')
print(f'Azimuth range : {AZIMUTHS[0]}° to {AZIMUTHS[-1]}°')
print(f'Elevation range: {ELEVATIONS[0]:.2f}° to {ELEVATIONS[-1]:.2f}°')

---
## 3. Load and Explore the HRTF Data

In [ ]:
# ── Load the .mat file ────────────────────────────────────────────────────────
hrir_data = scipy.io.loadmat(LOCAL_MAT)

# hrir_l and hrir_r have shape (N_AZ, N_EL, N_HRIR) = (25, 50, 200)
hrir_l = np.array(hrir_data['hrir_l'])   # Left ear impulse responses
hrir_r = np.array(hrir_data['hrir_r'])   # Right ear impulse responses

print('HRIR data loaded:')
print(f'  hrir_l shape: {hrir_l.shape}  → (n_az, n_el, n_samples)')
print(f'  hrir_r shape: {hrir_r.shape}')
print(f'  Sample rate : {FS} Hz')
print(f'  HRIR duration: {N_HRIR/FS*1000:.2f} ms per position')
print(f'  Value range : [{hrir_l.min():.4f}, {hrir_l.max():.4f}]')

In [ ]:
# ── Helper: get HRIR for a given (azimuth, elevation) pair ───────────────────
def get_hrir(az_deg, el_deg):
    """
    Return (hrir_left, hrir_right) for the closest measured position
    to the requested azimuth and elevation.
    """
    az_idx = np.argmin(np.abs(AZIMUTHS   - az_deg))
    el_idx = np.argmin(np.abs(ELEVATIONS - el_deg))
    return hrir_l[az_idx, el_idx], hrir_r[az_idx, el_idx]


def get_hrir_interp(az_deg, el_deg):
    """
    Return (hrir_left, hrir_right) using barycentric interpolation
    from the three nearest measured positions (Delaunay triangulation).
    """
    points  = np.array(list(itertools.product(AZIMUTHS, ELEVATIONS)))
    tri     = scipy.spatial.Delaunay(points)
    target  = np.array([[az_deg, el_deg]])
    simplex = tri.find_simplex(target)

    if simplex < 0:
        # Outside convex hull — fall back to nearest neighbour
        return get_hrir(az_deg, el_deg)

    vertices = tri.simplices[simplex[0]]   # three vertex indices
    b  = tri.transform[simplex[0], :2].dot(
             (target - tri.transform[simplex[0], 2]).T).T[0]
    weights = np.r_[b, 1 - b.sum()]       # barycentric coordinates

    h_l = np.zeros(N_HRIR)
    h_r = np.zeros(N_HRIR)
    for w, vi in zip(weights, vertices):
        az_i, el_i = points[vi]
        hl, hr = get_hrir(az_i, el_i)
        h_l += w * hl
        h_r += w * hr

    return h_l, h_r


# Quick test
hl0, hr0 = get_hrir(0, 0)   # straight ahead
hl90, hr90 = get_hrir(45, 0) # far right
print('Straight ahead (0°az, 0°el) — L/R peak magnitudes:', np.max(np.abs(hl0)), np.max(np.abs(hr0)))
print('Far right (45°az, 0°el)   — L/R peak magnitudes:', np.max(np.abs(hl90)), np.max(np.abs(hr90)))

---
## 4. Visualizing HRTFs

Let's explore what the measured impulse responses look like and how they change with direction.

In [ ]:
# ── 4.1 HRIR time-domain comparison for several azimuths ─────────────────────
t = np.arange(N_HRIR) / FS * 1000   # time axis in milliseconds
azimuths_to_plot = [-80, -45, 0, 45, 80]
colors = cm.coolwarm(np.linspace(0, 1, len(azimuths_to_plot)))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for az, c in zip(azimuths_to_plot, colors):
    hl, hr = get_hrir(az, 0)
    axes[0].plot(t, hl, color=c, alpha=0.85, label=f'{az:+d}°')
    axes[1].plot(t, hr, color=c, alpha=0.85, label=f'{az:+d}°')

for ax, title in zip(axes, ['Left ear HRIR', 'Right ear HRIR']):
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('Amplitude')
    ax.set_title(title)
    ax.legend(title='Azimuth', fontsize=9)

fig.suptitle('HRIRs at Elevation 0° — varying Azimuth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('🔍 Notice: for sounds from the RIGHT (positive azimuth),')
print('   the right-ear HRIR peaks EARLIER and is LARGER than the left — this is the ITD+ILD.')

In [ ]:
# ── 4.2 Magnitude spectrum (HRTF) — frequency-domain view ────────────────────
NFFT = 512
freqs = np.fft.rfftfreq(NFFT, d=1/FS)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for az, c in zip(azimuths_to_plot, colors):
    hl, hr = get_hrir(az, 0)
    Hl = np.abs(np.fft.rfft(hl, n=NFFT))
    Hr = np.abs(np.fft.rfft(hr, n=NFFT))
    axes[0].semilogx(freqs[1:], 20*np.log10(Hl[1:] + 1e-12), color=c, alpha=0.85, label=f'{az:+d}°')
    axes[1].semilogx(freqs[1:], 20*np.log10(Hr[1:] + 1e-12), color=c, alpha=0.85, label=f'{az:+d}°')

for ax, title in zip(axes, ['Left ear HRTF |H(f)|', 'Right ear HRTF |H(f)|']):
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_title(title)
    ax.set_xlim([200, FS/2])
    ax.set_ylim([-50, 10])
    ax.legend(title='Azimuth', fontsize=9)

fig.suptitle('HRTF Magnitude Spectra at Elevation 0°', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('🔍 Above ~4 kHz the head shadow creates a significant dip in the ear farther from the source (ILD).')

In [ ]:
# ── 4.3 Interaural Time Difference (ITD) map ──────────────────────────────────
# ITD = delay of the first peak between left and right ear
def estimate_itd(hl, hr):
    """Estimate ITD in microseconds via cross-correlation peak."""
    xcorr = scipy.signal.correlate(hl, hr, mode='full')
    lags  = scipy.signal.correlation_lags(len(hl), len(hr))
    itd_samples = lags[np.argmax(xcorr)]
    return itd_samples / FS * 1e6   # convert to µs

# Build ITD map over all azimuths and elevations
el_subset = ELEVATIONS[ELEVATIONS <= 90]   # front hemisphere for clarity
el_indices = [np.argmin(np.abs(ELEVATIONS - e)) for e in el_subset]

ITD_map = np.zeros((N_AZ, len(el_subset)))
for i, az in enumerate(AZIMUTHS):
    for j, ei in enumerate(el_indices):
        hl = hrir_l[i, ei]
        hr = hrir_r[i, ei]
        ITD_map[i, j] = estimate_itd(hl, hr)

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(ITD_map.T, aspect='auto', origin='lower',
               extent=[AZIMUTHS[0], AZIMUTHS[-1], el_subset[0], el_subset[-1]],
               cmap='RdBu', vmin=-700, vmax=700)
plt.colorbar(im, ax=ax, label='ITD (µs)  [positive = right ear first]')
ax.set_xlabel('Azimuth (°)')
ax.set_ylabel('Elevation (°)')
ax.set_title('Interaural Time Difference (ITD) Map — Subject 021 (KEMAR)', fontweight='bold')
ax.axvline(0, color='k', linewidth=1.5, linestyle='--', alpha=0.5)
ax.axhline(0, color='k', linewidth=1.5, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('🔍 ITD is symmetric: positive azimuth → right ear first (positive ITD, red).')
print('   Maximum ITD is ~700 µs, matching the theoretical ≈ 690 µs for a human head.')

In [ ]:
# ── 4.4 Interaural Level Difference (ILD) map ─────────────────────────────────
def estimate_ild(hl, hr, f_low=3000):
    """Estimate ILD in dB, using RMS above f_low Hz (high-frequency dominant cue)."""
    NFFT = 512
    Hl = np.abs(np.fft.rfft(hl, n=NFFT))
    Hr = np.abs(np.fft.rfft(hr, n=NFFT))
    freqs = np.fft.rfftfreq(NFFT, d=1/FS)
    mask  = freqs >= f_low
    rms_l = np.sqrt(np.mean(Hl[mask]**2)) + 1e-12
    rms_r = np.sqrt(np.mean(Hr[mask]**2)) + 1e-12
    return 20 * np.log10(rms_r / rms_l)

ILD_map = np.zeros((N_AZ, len(el_subset)))
for i in range(N_AZ):
    for j, ei in enumerate(el_indices):
        ILD_map[i, j] = estimate_ild(hrir_l[i, ei], hrir_r[i, ei])

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(ILD_map.T, aspect='auto', origin='lower',
               extent=[AZIMUTHS[0], AZIMUTHS[-1], el_subset[0], el_subset[-1]],
               cmap='RdBu', vmin=-20, vmax=20)
plt.colorbar(im, ax=ax, label='ILD (dB)  [positive = louder in right ear]')
ax.set_xlabel('Azimuth (°)')
ax.set_ylabel('Elevation (°)')
ax.set_title('Interaural Level Difference (ILD) Map — > 3 kHz', fontweight='bold')
ax.axvline(0, color='k', linewidth=1.5, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('🔍 ILD clearly shows the head shadow effect — sounds from the right are')
print('   attenuated at the left ear at high frequencies (> 3 kHz).')

---
## 5. Synthesizing Sound Sources

We will generate three mono audio sources from scratch:
1. 🔊 **White noise** — flat spectrum, useful for hearing spectral coloration
2. 🐕 **Dog bark** — amplitude-modulated noise bursts with low-frequency emphasis
3. 💎 **Glass breaking** — transient impact followed by high-frequency cascade

In [ ]:
rng = np.random.default_rng(seed=42)

def make_white_noise(duration=1.0, fs=FS, amplitude=0.25):
    """Band-limited white noise (300 Hz – 8 kHz) for clear HRTF colouration demo."""
    n = int(duration * fs)
    noise = rng.standard_normal(n) * amplitude
    sos = scipy.signal.butter(4, [300, 8000], btype='bandpass', fs=fs, output='sos')
    return scipy.signal.sosfiltfilt(sos, noise)


def make_dog_bark(duration=0.6, fs=FS):
    """
    Synthesise a convincing dog bark:
    - Two short noise bursts with a low-frequency resonance (~500 Hz)
    - Amplitude envelope: fast attack, moderate decay
    - Adds a harmonic buzz layer
    """
    t = np.linspace(0, duration, int(duration * fs))

    def bark_burst(t_start, t_end, fund_freq):
        mask  = (t >= t_start) & (t < t_end)
        burst = np.zeros_like(t)
        seg   = t[mask] - t_start
        seg_t = t_end - t_start
        env   = np.exp(-8 * (seg / seg_t))          # fast-decay envelope
        noise = rng.standard_normal(seg.size)
        sos   = scipy.signal.butter(3, [200, 2500], btype='bandpass', fs=fs, output='sos')
        noise = scipy.signal.sosfilt(sos, noise)
        tone  = 0.4 * np.sin(2 * np.pi * fund_freq * seg)
        burst[mask] = env * (0.6 * noise + tone)
        return burst

    signal  = bark_burst(0.00, 0.18, 480)
    signal += bark_burst(0.22, 0.40, 520)
    signal += bark_burst(0.44, 0.60, 500)
    signal  = signal / (np.max(np.abs(signal)) + 1e-9) * 0.5
    return signal


def make_glass_breaking(duration=1.2, fs=FS):
    """
    Synthesise glass breaking:
    - Short transient impact (high energy)
    - Followed by cascading high-frequency noise that decays
    - Multiple resonant modes
    """
    n = int(duration * fs)
    t = np.linspace(0, duration, n)

    # Initial impact
    impact = np.zeros(n)
    imp_end = int(0.01 * fs)
    impact[:imp_end] = rng.standard_normal(imp_end) * np.exp(-80 * np.linspace(0, 0.01, imp_end) / 0.01)

    # Cascading shards (several high-frequency resonances)
    shatter = np.zeros(n)
    for freq, amp, decay in [(4000, 0.4, 6), (6500, 0.3, 9), (9000, 0.2, 15), (3000, 0.3, 5)]:
        noise = rng.standard_normal(n)
        sos = scipy.signal.butter(3, [freq*0.8, min(freq*1.2, 20000)],
                                  btype='bandpass', fs=fs, output='sos')
        noise = scipy.signal.sosfilt(sos, noise)
        env   = np.exp(-decay * t)
        shatter += amp * env * noise

    # Longer low-amplitude tail
    tail = np.zeros(n)
    tail_noise = rng.standard_normal(n)
    sos = scipy.signal.butter(3, [2000, 12000], btype='bandpass', fs=fs, output='sos')
    tail_noise = scipy.signal.sosfilt(sos, tail_noise)
    tail = 0.15 * np.exp(-3 * t) * tail_noise

    signal  = 0.8 * impact + 0.5 * shatter + tail
    signal  = signal / (np.max(np.abs(signal)) + 1e-9) * 0.6
    return signal


# ── Generate all sources ──────────────────────────────────────────────────────
SOURCES = {
    'white_noise' : make_white_noise(1.5),
    'dog_bark'    : make_dog_bark(),
    'glass'       : make_glass_breaking(),
}

# Preview each source (mono, no HRTF yet)
print('=== Sound Source Previews (MONO) ===')
for name, sig in SOURCES.items():
    print(f'\n▶ {name} ({len(sig)/FS:.2f}s):')
    display(Audio(sig, rate=FS, normalize=True))

In [ ]:
# ── Waveform visualisation of the three sources ───────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=False)
titles = {'white_noise': '🔊 White Noise', 'dog_bark': '🐕 Dog Bark', 'glass': '💎 Glass Breaking'}
colors_s = ['steelblue', 'darkorange', 'mediumseagreen']

for ax, (name, sig), col in zip(axes, SOURCES.items(), colors_s):
    t_sig = np.arange(len(sig)) / FS
    ax.plot(t_sig, sig, lw=0.6, color=col, alpha=0.9)
    ax.set_title(titles[name], fontweight='bold', loc='left')
    ax.set_ylabel('Amplitude')
    ax.set_ylim([-0.7, 0.7])

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Synthesized Sound Sources (Mono)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Binaural Rendering Engine

Convolution of a mono signal with an HRIR pair produces a binaural stereo signal. We use **overlap-add** via `scipy.signal.fftconvolve` for efficiency.

In [ ]:
def spatialize(mono, az_deg, el_deg, use_interp=True, fade_ms=5, fs=FS):
    """
    Apply HRTF spatialization to a mono audio signal.

    Parameters
    ----------
    mono       : 1-D numpy array — mono input signal
    az_deg     : float — desired azimuth in degrees
    el_deg     : float — desired elevation in degrees
    use_interp : bool  — use barycentric interpolation (True) or nearest neighbour
    fade_ms    : float — fade-in/fade-out in ms to avoid clicks
    fs         : int   — sample rate

    Returns
    -------
    binaural   : (N, 2) float32 array — [left, right] stereo output
    """
    if use_interp:
        hl, hr = get_hrir_interp(az_deg, el_deg)
    else:
        hl, hr = get_hrir(az_deg, el_deg)

    # Convolve (FFT-based for speed)
    out_l = scipy.signal.fftconvolve(mono, hl)
    out_r = scipy.signal.fftconvolve(mono, hr)

    # Trim to same length as input + HRIR tail
    N_out = len(mono) + len(hl) - 1
    out_l = out_l[:N_out]
    out_r = out_r[:N_out]

    # Fade in/out
    fade_n = int(fade_ms * fs / 1000)
    if fade_n > 0:
        fade = np.linspace(0, 1, fade_n)
        out_l[:fade_n]  *= fade; out_l[-fade_n:] *= fade[::-1]
        out_r[:fade_n]  *= fade; out_r[-fade_n:] *= fade[::-1]

    # Normalize
    peak = max(np.max(np.abs(out_l)), np.max(np.abs(out_r)), 1e-9)
    out_l /= peak * 1.1
    out_r /= peak * 1.1

    return np.column_stack([out_l, out_r]).astype(np.float32)


def audio_to_bytes(binaural, fs=FS):
    """Convert a (N,2) float array to in-memory WAV bytes for IPython.display.Audio."""
    buf = io.BytesIO()
    sf.write(buf, binaural, fs, format='WAV', subtype='PCM_16')
    buf.seek(0)
    return buf.read()


# ── Verify: spatialize white noise from the right side ────────────────────────
test_binaural = spatialize(SOURCES['white_noise'], az_deg=45, el_deg=0)
print(f'Output shape: {test_binaural.shape}  (samples × 2 channels)')
print(f'Left  RMS: {np.sqrt(np.mean(test_binaural[:,0]**2)):.4f}')
print(f'Right RMS: {np.sqrt(np.mean(test_binaural[:,1]**2)):.4f}')
print('→ Right RMS should be larger for az=+45° (sound from the right)')

In [ ]:
# ── Static demo: hear a sound from three different positions ──────────────────
demo_positions = [
    (0,   0,  'Straight ahead  (0°, 0°)'),
    (80,  0,  'Far right       (+80°, 0°)'),
    (-80, 0,  'Far left        (−80°, 0°)'),
    (0,   90, 'Above           (0°, 90°)'),
]

print('=== 🐕 Dog Bark — Spatialized Demo ===')
print('⚠️  Use headphones for the 3D effect!\n')
for az, el, label in demo_positions:
    binaural = spatialize(SOURCES['dog_bark'], az, el)
    print(f'▶ {label}')
    display(Audio(binaural.T, rate=FS, normalize=False))

---
## 7. 🎮 Interactive Spatial Audio Explorer

Use the sliders below to position the sound in 3D space and press **"🔊 Render & Play"** to hear it.

> **Coordinate system reminder**:  
> - **Azimuth** = left/right (−80° = far left, 0° = straight ahead, +80° = far right)  
> - **Elevation** = up/down/behind (−45° = below, 0° = horizontal, 90° = above, 180° = behind)

In [ ]:
# ── VS Code / Jupyter widget setup ───────────────────────────────────────────
# Run this cell ONCE before the interactive demo.
# In VS Code: make sure the Python environment has ipywidgets installed,
# and that the Jupyter extension is up to date (≥ v2024).
#
# If sliders are still invisible after running this cell, try:
#   pip install --upgrade ipywidgets
# then: Kernel → Restart Kernel and Run All Cells

import subprocess, sys
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, fixed
    print(f'✅ ipywidgets {widgets.__version__} ready')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets'])
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, fixed
    print('✅ ipywidgets installed and ready')

# Verify widget rendering works in this environment
test_slider = widgets.IntSlider(value=5, min=0, max=10, description='Test:')
display(test_slider)
print('☝️  You should see a blue slider above. If not, restart the kernel.')


In [ ]:
# ── 🎮 Interactive Spatial Audio Explorer ────────────────────────────────────
# Uses @widgets.interact — the most VS Code-compatible ipywidgets approach.
# Sliders update the head diagram; press the Render button to hear the audio.
#
# ⚠️  HEADPHONES REQUIRED for the 3D effect.

import base64, io

# ── Head diagram helper (returns inline PNG for Output widget) ────────────────
def draw_head_diagram(az, el):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

    for ax, (x_src, y_src, xtxt, ytxt, front_lbl, back_lbl, l_lbl, r_lbl, title) in zip(
        axes,
        [
            # Horizontal (azimuth) view
            (np.sin(np.deg2rad(az)), np.cos(np.deg2rad(az)),
             np.sin(np.deg2rad(az))*1.55, np.cos(np.deg2rad(az))*1.55,
             'FRONT', 'BACK', 'L', 'R',
             f'Top view — Azimuth {az:+d}°'),
            # Vertical (elevation) view
            (np.sin(np.deg2rad(el)), np.cos(np.deg2rad(el)),
             np.sin(np.deg2rad(el))*1.55, np.cos(np.deg2rad(el))*1.55,
             'UP', 'DOWN', '', '',
             f'Side view — Elevation {el:.1f}°'),
        ]
    ):
        ring = plt.Circle((0, 0), 1, color='#E8EEF4', fill=True, zorder=1)
        ax.add_patch(ring)
        ax.plot([0, 0], [0, 1.1], 'k-', lw=1.2, alpha=0.25)
        ax.scatter([0], [0], s=260, c='#3a86ff', zorder=5)
        ax.scatter([x_src * 1.4], [y_src * 1.4], s=220,
                   c='#ff6b35', marker='*', zorder=6)
        ax.text(xtxt, ytxt, f'★', ha='center', va='center',
                fontsize=14, color='#ff6b35')
        ax.text(0,  1.28, front_lbl, ha='center', va='center', fontsize=8, alpha=0.5)
        ax.text(0, -1.32, back_lbl,  ha='center', va='center', fontsize=8, alpha=0.5)
        if l_lbl: ax.text(-1.38, 0, l_lbl, ha='center', va='center', fontsize=9, fontweight='bold')
        if r_lbl: ax.text( 1.38, 0, r_lbl, ha='center', va='center', fontsize=9, fontweight='bold')
        ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.8, 1.8)
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(title, fontweight='bold', fontsize=10)

    plt.tight_layout()
    plt.show()


# ── Render + play function ────────────────────────────────────────────────────
def render_and_play(source, azimuth, elevation, interpolate):
    """
    Spatialize the selected source and display the binaural audio player.
    Called automatically by @interact when any slider moves.
    """
    print(f'Rendering "{source}" at az={azimuth:+d}°, el={elevation:.2f}°  …', end=' ')
    binaural = spatialize(SOURCES[source], azimuth, elevation,
                           use_interp=interpolate)
    print('done ✅  — 🎧 use headphones!')

    # Head diagram
    draw_head_diagram(azimuth, elevation)

    # Waveform
    fig, axes = plt.subplots(2, 1, figsize=(11, 3.5), sharex=True)
    t_b = np.arange(len(binaural)) / FS
    axes[0].plot(t_b, binaural[:, 0], lw=0.6, color='cornflowerblue')
    axes[0].set_ylabel('Left ear')
    axes[0].set_title(f'Binaural output — {source}  |  az={azimuth:+d}°, el={elevation:.1f}°',
                      fontweight='bold')
    axes[1].plot(t_b, binaural[:, 1], lw=0.6, color='tomato')
    axes[1].set_ylabel('Right ear'); axes[1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()

    # Audio player
    display(Audio(binaural.T, rate=FS, normalize=False))


# ── Widget controls ───────────────────────────────────────────────────────────
source_widget = widgets.ToggleButtons(
    options=[('🔊 White Noise', 'white_noise'),
             ('🐕 Dog Bark',    'dog_bark'),
             ('💎 Glass',       'glass')],
    value='dog_bark',
    description='Source:',
    style={'description_width': '60px', 'button_width': '130px'}
)
az_widget = widgets.SelectionSlider(
    options=list(AZIMUTHS),
    value=0,
    description='Azimuth (°)',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='500px')
)
el_widget = widgets.SelectionSlider(
    options=[round(e, 3) for e in ELEVATIONS.tolist()],
    value=0.0,
    description='Elevation (°)',
    style={'description_width': '110px'},
    layout=widgets.Layout(width='500px')
)
interp_widget = widgets.Checkbox(
    value=True, description='Interpolate HRTF between grid points',
    style={'description_width': '220px'}
)

display(widgets.HTML('<h3 style="color:#2a5caa; margin-bottom:4px;">🎧 Spatial Audio Explorer</h3>'))
display(widgets.HTML(
    '<p style="color:#666; font-size:0.9em;">'
    'Adjust the sliders and the audio player will render automatically. '
    'Wear <strong>headphones</strong> for the 3D effect.</p>'
))

# @interact wires sliders → function automatically; works in VS Code, JupyterLab, classic
out = widgets.interactive(
    render_and_play,
    source=source_widget,
    azimuth=az_widget,
    elevation=el_widget,
    interpolate=interp_widget,
)
# interactive() returns a VBox with controls on top — display just the control area
# so VS Code renders it as a flat list of widgets (most compatible).
controls = widgets.VBox(out.children[:-1])  # all children except the Output
output   = out.children[-1]
display(controls, output)
out.update()   # trigger initial render


---
## 8. Moving Sound Source — Circular Trajectory

Render a sound source moving continuously around the listener in the horizontal plane.

In [ ]:
def render_moving_source(source_name, elevation=0.0,
                          n_steps=8, crossfade_ms=25, fs=FS):
    """
    Render a sound source orbiting in the horizontal plane.
    The azimuth visits all 25 CIPIC positions in order (left → right → left).

    Parameters
    ----------
    source_name  : key in SOURCES dict
    elevation    : fixed elevation in degrees
    n_steps      : number of orbit steps (uses every nth azimuth)
    crossfade_ms : crossfade duration between segments in milliseconds
    """
    mono = SOURCES[source_name]
    # Create a set of azimuths to visit: full orbit (L→R→L)
    orbit = list(AZIMUTHS) + list(AZIMUTHS[::-1])
    orbit_sampled = orbit[::max(1, len(orbit)//n_steps)][:n_steps]

    seg_len   = len(mono) // len(orbit_sampled)
    xfade_n   = int(crossfade_ms * fs / 1000)
    total_len = seg_len * len(orbit_sampled) + N_HRIR

    out_l = np.zeros(total_len)
    out_r = np.zeros(total_len)

    for k, az in enumerate(orbit_sampled):
        start = k * seg_len
        mono_seg = mono[start:start + seg_len]
        if len(mono_seg) == 0:
            break

        hl, hr = get_hrir(az, elevation)
        seg_l  = scipy.signal.fftconvolve(mono_seg, hl)
        seg_r  = scipy.signal.fftconvolve(mono_seg, hr)

        # Fade in/out for smooth crossfade
        fade_in  = np.linspace(0, 1, min(xfade_n, len(seg_l)))
        fade_out = fade_in[::-1]
        seg_l[:len(fade_in)] *= fade_in;   seg_l[-len(fade_out):] *= fade_out
        seg_r[:len(fade_in)] *= fade_in;   seg_r[-len(fade_out):] *= fade_out

        end_l = min(start + len(seg_l), total_len)
        end_r = min(start + len(seg_r), total_len)
        out_l[start:end_l] += seg_l[:end_l - start]
        out_r[start:end_r] += seg_r[:end_r - start]

    peak = max(np.max(np.abs(out_l)), np.max(np.abs(out_r)), 1e-9)
    out_l /= peak * 1.1
    out_r /= peak * 1.1

    return np.column_stack([out_l, out_r]).astype(np.float32), orbit_sampled


print('Rendering orbit (white noise sweeping left → right → left)…')
orbit_audio, orbit_azimuths = render_moving_source('white_noise', elevation=0.0, n_steps=12)

print(f'Orbit azimuths: {orbit_azimuths}')
print(f'Output duration: {len(orbit_audio)/FS:.2f}s')
print('\n▶ Orbit audio (🎧 headphones required):')
display(Audio(orbit_audio.T, rate=FS, normalize=False))

# Trajectory plot
fig, ax = plt.subplots(figsize=(8, 4))
t_orbit = np.linspace(0, len(orbit_audio)/FS, len(orbit_azimuths)+1)[:-1]
ax.step(t_orbit, orbit_azimuths, where='post', color='royalblue', lw=2)
ax.scatter(t_orbit, orbit_azimuths, c=orbit_azimuths, cmap='coolwarm', s=80, zorder=5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Azimuth (°)')
ax.set_title('Sound Source Trajectory — Horizontal Orbit', fontweight='bold')
ax.axhline(0, color='k', lw=1, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

### 8b. Smooth Orbit — Time-Varying HRTF via Overlap-Add

The abruptness in the previous cell comes from switching an entire HRIR block at once. The fix is **time-varying convolution** via the overlap-add (OLA) method:

1. **Short analysis frames** (512 samples ≈ 11 ms) — much shorter than any perceptible position change
2. **Hann window** applied to each frame so adjacent frames blend seamlessly
3. **HRIR interpolated** at the exact position for *each* frame using barycentric interpolation
4. **Overlap-add** reconstruction: frames are shifted by `hop_size` (256 samples, 50% overlap) and summed

The trajectory is a smooth **sinusoid** (−80° → +80° → −80°) instead of discrete jumps:

```
azimuth(t) = 80 · sin(2π · t / T)
```

For each frame at time `t`, the HRIR used is:

```
h(t) = α · h(az₁) + (1−α) · h(az₂)
```

where `az₁`, `az₂` are the two nearest measured azimuths and `α` is the fractional distance between them.

> **Why does this sound smoother?** Because the binaural cues (ITD and ILD) transition gradually — the brain perceives continuous motion rather than discrete jumps.

In [ ]:
# ── 8b. Smooth Orbit via Overlap-Add Time-Varying Convolution ─────────────────

def render_smooth_orbit(source_name, elevation=0.0, n_laps=1,
                         frame_size=512, hop_size=256, fs=FS):
    """
    Render a source orbiting the listener using frame-by-frame HRTF interpolation.

    The key difference from render_moving_source():
    - Frame size is 512 samples (≈ 11 ms) instead of large segment chunks
    - Azimuth follows a smooth sinusoid, not discrete steps
    - The HRIR is *linearly interpolated* between the two nearest measured
      azimuths at every single frame — ~100 updates per second
    - Hann-windowed overlap-add (50% overlap) ensures smooth reconstruction

    Parameters
    ----------
    source_name : key in SOURCES dict
    elevation   : fixed elevation in degrees
    n_laps      : number of full orbits (L→R→L = 1 lap)
    frame_size  : OLA frame length in samples (smaller = smoother, more CPU)
    hop_size    : OLA hop in samples (frame_size // 2 = 50% overlap)
    """
    mono = SOURCES[source_name]
    N    = len(mono)

    # ── Smooth sinusoidal azimuth trajectory ─────────────────────────────────
    # Number of frames needed to cover the whole signal
    n_frames  = int(np.ceil(N / hop_size)) + 1
    t_norm    = np.linspace(0, n_laps * 2 * np.pi, n_frames)
    az_traj   = 80.0 * np.sin(t_norm)      # −80° → 0° → +80° → 0° → −80°

    # ── Overlap-add buffers ───────────────────────────────────────────────────
    out_len = N + N_HRIR + frame_size
    out_l   = np.zeros(out_len)
    out_r   = np.zeros(out_len)

    # Hann window — ensures smooth fade-in/fade-out at each frame boundary
    window = np.hanning(frame_size)

    # ── Per-frame interpolated convolution ────────────────────────────────────
    for i in range(n_frames):
        start = i * hop_size
        if start >= N:
            break

        # Extract frame (zero-pad at end of signal)
        frame = np.zeros(frame_size)
        avail = min(frame_size, N - start)
        frame[:avail] = mono[start:start + avail]
        frame *= window

        # Interpolate HRIR at the exact azimuth for this frame
        az_now = az_traj[i]

        # Find the two nearest measured azimuths and their fractional weight
        dists = np.abs(AZIMUTHS - az_now)
        idx   = np.argsort(dists)
        i0, i1 = idx[0], idx[1]
        az0, az1 = AZIMUTHS[i0], AZIMUTHS[i1]

        # Linear interpolation weight (0 = fully az0, 1 = fully az1)
        span  = abs(az1 - az0) if az0 != az1 else 1.0
        alpha = abs(az_now - az0) / span
        alpha = np.clip(alpha, 0.0, 1.0)

        el_idx = np.argmin(np.abs(ELEVATIONS - elevation))
        hl = (1 - alpha) * hrir_l[i0, el_idx] + alpha * hrir_l[i1, el_idx]
        hr = (1 - alpha) * hrir_r[i0, el_idx] + alpha * hrir_r[i1, el_idx]

        # Convolve windowed frame with interpolated HRIR
        seg_l = scipy.signal.fftconvolve(frame, hl)
        seg_r = scipy.signal.fftconvolve(frame, hr)

        # Overlap-add into output buffer
        end_l = min(start + len(seg_l), out_len)
        end_r = min(start + len(seg_r), out_len)
        out_l[start:end_l] += seg_l[:end_l - start]
        out_r[start:end_r] += seg_r[:end_r - start]

    # Trim and normalise
    out_l = out_l[:N + N_HRIR]
    out_r = out_r[:N + N_HRIR]
    peak  = max(np.max(np.abs(out_l)), np.max(np.abs(out_r)), 1e-9)
    out_l /= peak * 1.1
    out_r /= peak * 1.1

    return np.column_stack([out_l, out_r]).astype(np.float32), az_traj


# ── Render and compare ────────────────────────────────────────────────────────
print('Rendering ABRUPT orbit (original)  …', end=' ', flush=True)
abrupt_audio, abrupt_azimuths = render_moving_source('white_noise', elevation=0.0, n_steps=12)
print('done')

print('Rendering SMOOTH orbit (OLA)        …', end=' ', flush=True)
smooth_audio, smooth_az_traj = render_smooth_orbit('white_noise', elevation=0.0, n_laps=1)
print('done')

# ── Side-by-side trajectory plots ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Abrupt: step function
t_ab = np.linspace(0, len(abrupt_audio) / FS, len(abrupt_azimuths) + 1)[:-1]
axes[0].step(t_ab, abrupt_azimuths, where='post', color='tomato', lw=2)
axes[0].scatter(t_ab, abrupt_azimuths, c='tomato', s=60, zorder=5)
axes[0].axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Azimuth (°)')
axes[0].set_title('Abrupt — discrete step switches', fontweight='bold')
axes[0].set_ylim(-95, 95)

# Smooth: sinusoidal
t_sm = np.linspace(0, len(smooth_audio) / FS, len(smooth_az_traj))
axes[1].plot(t_sm, smooth_az_traj, color='steelblue', lw=2)
axes[1].fill_between(t_sm, smooth_az_traj, alpha=0.15, color='steelblue')
axes[1].axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.4)
axes[1].set_xlabel('Time (s)')
axes[1].set_title('Smooth OLA — sinusoidal, interpolated HRIR per frame', fontweight='bold')
axes[1].set_ylim(-95, 95)

fig.suptitle('Source Trajectory: Abrupt vs. Smooth', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Audio players ─────────────────────────────────────────────────────────────
print('\n▶ ABRUPT orbit (🔴 notice the jumps):  🎧 headphones required')
display(Audio(abrupt_audio.T, rate=FS, normalize=False))

print('\n▶ SMOOTH OLA orbit (🔵 continuous motion):')
display(Audio(smooth_audio.T, rate=FS, normalize=False))

# ── Frame-rate information ────────────────────────────────────────────────────
hop_ms  = 256 / FS * 1000
n_upd   = len(smooth_az_traj)
print(f'\n📊 HRTF update rate: every {hop_ms:.1f} ms  ({1000/hop_ms:.0f} updates/s)')
print(f'   Total HRIR interpolations computed: {n_upd}')
print(f'   Trajectory: sinusoid from −80° to +80° and back, 1 full orbit')


---
## 9. Spectral Analysis: How HRTF Colours the Sound

Compare the spectrogram of a spatialized signal with the mono original.

In [ ]:
# Choose a position and source to analyse
ANALYSE_AZ = 45
ANALYSE_EL = 0
ANALYSE_SRC = 'white_noise'

mono_sig   = SOURCES[ANALYSE_SRC]
binaural   = spatialize(mono_sig, ANALYSE_AZ, ANALYSE_EL)
binaural_l = binaural[:len(mono_sig), 0]
binaural_r = binaural[:len(mono_sig), 1]

# Compute spectrograms with librosa
HOP = 512; WIN = 2048

def make_spec(sig):
    S = np.abs(librosa.stft(sig.astype(float), n_fft=WIN, hop_length=HOP))
    return librosa.amplitude_to_db(S, ref=np.max)

S_mono = make_spec(mono_sig)
S_l    = make_spec(binaural_l)
S_r    = make_spec(binaural_r)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, S, title in zip(axes, [S_mono, S_l, S_r],
                         ['Mono (original)', f'Left ear  (az={ANALYSE_AZ:+d}°)',
                          f'Right ear (az={ANALYSE_AZ:+d}°)']):
    librosa.display.specshow(S, sr=FS, hop_length=HOP, x_axis='time',
                             y_axis='log', ax=ax, cmap='magma', vmin=-60)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Time (s)')

fig.suptitle('Spectrogram: HRTF Spectral Coloration Effect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('🔍 Notice the energy difference between left and right ears above ~4 kHz.')
print(f'   Sound from the right ({ANALYSE_AZ}°) → right ear brighter, left ear shadowed.')

---
## 10. Exercises for Students

### 10.1 — Experiment with Different Subjects
Change `SUBJECT = '021'` in the download cell to another subject ID (e.g. `'003'`, `'008'`, `'165'`) and compare how the ITD and ILD maps differ between individuals.

### 10.2 — Cone of Confusion
The **cone of confusion** is a set of positions that produce identical ITD and ILD (a cone around the interaural axis). Use the ITD and ILD maps above to identify positions where both cues are ambiguous. How does the pinna's spectral shaping help resolve the ambiguity?

### 10.3 — Custom Audio File
Run the cell below to load your own mono WAV file and spatialize it at any position.

In [ ]:
# ── 10.3: Load your own mono WAV and spatialize it ───────────────────────────
# Replace the path below with your WAV file (mono, any sample rate — librosa resamples)

MY_FILE = 'your_sound.wav'    # ← change this path
MY_AZ   = 30                  # azimuth
MY_EL   = 0                   # elevation

if os.path.exists(MY_FILE):
    sig, sr = librosa.load(MY_FILE, sr=FS, mono=True)
    print(f'Loaded: {MY_FILE} — {len(sig)/FS:.2f}s at {FS} Hz')
    binaural_custom = spatialize(sig, MY_AZ, MY_EL)
    print(f'\n▶ Your sound at az={MY_AZ:+d}°, el={MY_EL}°:')
    display(Audio(binaural_custom.T, rate=FS, normalize=False))
else:
    print(f'File not found: {MY_FILE}')
    print('Set MY_FILE to the path of a mono WAV file on your machine.')

---
## 11. Summary

In this notebook we:

1. **Explained** how HRTFs encode spatial hearing cues (ITD, ILD, spectral cues)
2. **Downloaded** and loaded the CIPIC database for subject 021 (KEMAR) using `scipy.io.loadmat`
3. **Visualized** HRIRs in the time domain, HRTFs in the frequency domain, and computed ITD/ILD maps
4. **Synthesized** three distinct sound sources (white noise, dog bark, glass breaking) from scratch using `numpy` and `scipy.signal`
5. **Implemented** binaural rendering via FFT convolution (`scipy.signal.fftconvolve`) with barycentric interpolation between measured positions
6. **Built** an interactive explorer using `ipywidgets` and `IPython.display.Audio` for real-time position adjustment
7. **Rendered** a moving sound source and visualized the resulting spectral coloration with `librosa`

### Key Takeaways

- **Binaural synthesis is linear**: it reduces to convolution, making it computationally straightforward
- **Individuality matters**: HRTFs are highly person-specific; a mismatch causes degraded externalization
- **Elevation cues are fragile**: unlike azimuth (ITD-dominant), elevation relies on pinna spectral cues which degrade with non-individual HRTFs
- **Headphones are essential**: HRTF rendering requires independent channel delivery to each ear

### References

- Algazi, V.R., Duda, R.O., Thompson, D.M., Avendano, C. (2001). *The CIPIC HRTF Database.* IEEE WASPAA.
- Blauert, J. (1997). *Spatial Hearing: The Psychophysics of Human Sound Localization.* MIT Press.
- Gamper, H. (2013). *Head-related transfer function interpolation in azimuth, elevation, and distance.* JASA 134(6).
- CIPIC Database: https://www.ece.ucdavis.edu/cipic/spatial-sound/hrtf-data/
- librosa: https://librosa.org
- scipy.signal: https://docs.scipy.org/doc/scipy/reference/signal.html